In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/abraham/uni/ikt453/project/v1

/home/abraham/uni/ikt453/project/v1


In [ ]:
from collections import defaultdict
from copy import deepcopy
from uuid import uuid4
from tqdm import tqdm

from src.utils import disk
from src.utils import debug

In [ ]:
def normalize_player_data(json_path: str) -> dict:
    data = disk.read_json(json_path)

    data = data['data']['props']['pageProps']['player']
    info = data['info']

    info = {k.lower(): v for k, v in info.items()}
    for key in info.keys():
        if key.endswith('_flag'):
            info[key] = {'Y': True, 'N': False}.get(info[key], info[key])
    
    return {
        '_meta': {
            'keys_processed': 'info',
            'keys_available': ', '.join(data.keys()),
            'path': json_path,
        },
        **info,
    }, data['roster']

In [ ]:
from _v2.utils.provenance import build_provenance_envelope
input_path = '_data/000_raw/nba/players/data'
output_path = '_data/001_staged/players/000_normalized/from_nba_players.json'

outputs = []
# for path in tqdm(disk.listdir(input_path)):
#     normalized = normalize_player_data(path)
#     outputs.append(normalized)

# disk.write_json(output_path, build_provenance_envelope(
#     is_directory_input=True,
#     is_directory_output=False,
#     path_input_data=input_path,
#     path_processing_script='_v2/normalize/002_players.ipynb',
#     data=outputs,
# ))

100%|██████████| 4390/4390 [00:03<00:00, 1436.41it/s]


In [46]:
def normalize_roster_data(json_path: str) -> dict:
    data = disk.read_json(json_path)
    data = data['data']['props']['pageProps']['player']
    return data['roster']

In [ ]:
# import pandas as pd
# from _v2.utils.provenance import build_provenance_envelope
# input_path = '_data/000_raw/nba/players/data'
# output_path = '_data/001_staged/teams/000_normalized/from_nba_players.json'
# directory, _ = disk.os.path.split(output_path)
# disk.makedirs(directory, exist_ok=True)

# outputs = []
# for path in tqdm(disk.listdir(input_path)):
#     normalized = normalize_roster_data(path)
#     outputs.extend(normalized)

# outputs = pd.DataFrame(outputs).drop_duplicates()
# outputs.columns = outputs.columns.str.lower()
# outputs = outputs.to_dict(orient='records')

# disk.write_json(output_path, build_provenance_envelope(
#     is_directory_input=True,
#     is_directory_output=False,
#     path_input_data=input_path,
#     path_processing_script='_v2/normalize/002_players.ipynb',
#     data=outputs,
# ))

100%|██████████| 4390/4390 [00:03<00:00, 1117.06it/s]


In [42]:
import pandas as pd
input_path = '_data/000_raw/nba/players/historic/player_index.json'
output_path = '_data/001_staged/players/000_normalized/from_nba_players_historic.json'
data = disk.read_json(input_path)
data = data['resultSets'][0]
data = pd.DataFrame(data['rowSet'], columns=data['headers'])
data = data.astype(object)
data = data.where(pd.notnull(data), None)
data.columns = data.columns.str.lower()
data = data.to_dict(orient='records')

envelope = build_provenance_envelope(
    is_directory_input=False,
    is_directory_output=False,
    path_input_data=input_path,
    path_processing_script='_v2/normalize/002_players.ipynb',
    data=data,
)

disk.write_json(output_path, envelope)

In [45]:
# def create_envelope(data):
#     return build_provenance_envelope(
#         source='sportradar',
#         path_input_data=None,
#         path_processing_script='_v2/sources/sportradar/download/team_profiles.py',
#         is_directory_output=False,
#         is_directory_input=False,
#         data=data,
#     )

# paths = '_data/000_raw/sportradar/teams/data'
# for path in disk.listdir(paths):
#     if disk.isfile(path):
#         envelope = create_envelope(disk.read_json(path))
#         disk.write_json(path, envelope)
#     elif disk.isdir(path):
#         for subpath in disk.listdir(path):
#             if disk.isfile(subpath):
#                 envelope = create_envelope(disk.read_json(subpath))
#                 disk.write_json(subpath, envelope)
    


In [17]:
import pandas as pd

outputs = disk.read_json(output_path)
df = pd.DataFrame(outputs)

In [18]:
info = pd.DataFrame(df['info'].tolist())

In [83]:
info.iloc[0]

person_id                                        201192
first_name                                      Taurean
last_name                                         Green
display_first_last                        Taurean Green
display_last_comma_first                 Green, Taurean
display_fi_last                                T. Green
player_slug                               taurean-green
birthdate                           1986-11-28T00:00:00
school                                          Florida
country                                             USA
last_affiliation                            Florida/USA
height                                              6-0
weight                                              177
season_exp                                            1
jersey                                                0
position                                          Guard
rosterstatus                                   Inactive
games_played_current_season_flag                

### Debugging

In [82]:
# info

In [20]:
from src.clients import connect_mysql

In [76]:
out = pd.read_sql('SELECT player_id, team_id, game_id FROM fact_player_game_stats', connect_mysql())

/tmp/ipykernel_3684724/2174010994.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql('SELECT player_id, team_id, game_id FROM fact_player_game_stats', connect_mysql())


In [31]:
out_p = pd.read_sql('SELECT player_id, first_name, family_name FROM dim_player', connect_mysql())

/tmp/ipykernel_3684724/1755681317.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out_p = pd.read_sql('SELECT player_id, first_name, family_name FROM dim_player', connect_mysql())


In [34]:
mask = True
mask &= out_p.first_name.str.strip() != ''
mask &= out_p.family_name.str.strip() != ''
out_p = out_p[mask]

In [47]:
out_p[out_p.player_id == '196294755']

,player_id,first_name,family_name
2203,196294755,Mecole,Hardman Jr.


In [48]:
out[out.player_id == '196294755']

,player_id,game_id
507701,196294755,0032300003


In [22]:
len(df)

4390

In [50]:
len(info), len(pidx)

(4390, 5126)

In [60]:
set_api = set(info.person_id.astype(str).unique())
set_idx = set(pidx.PERSON_ID.astype(str).unique())
diff = set_idx.difference(set_api)
disk.write_json('data/remainding_player_ids.json', list(diff))

In [44]:
set_a = set(out_p.player_id.unique())
set_b = set(info['person_id'].astype(str).unique())
set_c = set(pidx['PERSON_ID'].astype(str).unique())

In [66]:
# set_a.difference(set_c)

In [73]:
tricodes = '_data/001_staged/teams/002_support/tricode_current.json'
tricodes = disk.read_json(tricodes)
tricodes = pd.DataFrame(tricodes)

In [74]:
tricodes

,team_id,current_tricode,is_nba_team
0,41,MTA,False
1,45,CHN,False
2,93,MAC,False
3,94,MLN,False
4,95,LAB,False
...,...,...,...
70,1610612764,WAS,True
71,1610612765,DET,True
72,1610612766,CHA,True
73,1610616833,EST,False


In [80]:
m = out.player_id.isin(set_c)
t = out[~m].team_id.unique()
m = tricodes.team_id.isin(t)
t

array(['1610612738', '1610612761', '1610612750', '1610612747',
       '1610612764', '1610612762', '1610612756', '12308', '1610612739',
       '1610612765', '1610612742', '1610612760', '1610612754',
       '1610612748', '1610612745', '1610612753', '1610612741',
       '1610612746', '1610612749', '1610612740', '1610612759',
       '1610612751', '1610612752', '1610612758', '1610612766', '12304',
       '1610612757', '1610612755', '1610612744', '1610612743',
       '1610612737', '1610612763', '93', '1610616844', '1610616840',
       '1610616839', '15020', '50009', '12328', '12319', '1610616858',
       '15017', '12321', '41', '15015', '12320', '12317', '12315',
       '15019', '12322', '12313', '12329', '15018', '15016', '15022',
       '12325', '12318', '12330', '1610612767', '1610612778', '12316',
       '12323', '12303', '45', '1610616850', '12324', '95', '15025',
       '1610616833', '12309', '1610616837', '104', '12311', '12332',
       '1610616848', '94', '15021', '1610616834', '1231

In [45]:
from _v2.utils.eda import set_comparison

debug.prettyprint(set_comparison(set_a, set_c))

{
    "x": 3677,
    "y": 5126,
    "x_diff_y": 1153,
    "y_diff_x": 2602,
    "total": 6279,
    "same": 2524,
    "diff": 3755
}
